# Паттерн: Подграфы — агент как подграф (Subgraphs)

Любой скомпилированный граф LangGraph можно встроить как узел в другой граф. Это мощный механизм инкапсуляции: подграф имеет собственное внутреннее состояние и логику, а родительский граф видит только его входы и выходы.

В этом примере внутренний граф — двухшаговый агент (`process` -> `format`), который анализирует тему и оформляет результат. Внешний граф оборачивает его: подготавливает данные (`prepare`), вызывает внутренний агент как единый узел, а затем строит финальное резюме (`summarize`).

LangGraph автоматически сопоставляет одноимённые поля между состояниями родительского и дочернего графа — данные передаются без ручного маппинга.

```mermaid
graph LR
    subgraph outer["Внешний граф (OuterState)"]
        START((START)) --> prepare(["🔹 Prepare<br/><i>подготавливает данные</i>"])
        prepare --> agent

        subgraph agent["Подграф-агент (InnerState)"]
            direction TB
            process(["🔹 Process<br/><i>анализирует тему</i>"]) --> format(["🔹 Format<br/><i>оформляет результат</i>"])
        end

        agent --> summarize(["📊 Summarize<br/><i>агрегирует</i>"])
        summarize --> END((END))
    end

    classDef terminal fill:#95A5A6,stroke:#707B7C,color:#fff
    classDef worker fill:#2ECC71,stroke:#1A8B4C,color:#fff,stroke-width:2px
    classDef aggregator fill:#F59E0B,stroke:#D97706,color:#fff,stroke-width:2px
    classDef subgraphStyle fill:#FDEBD0,stroke:#E67E22,stroke-width:2px,color:#333

    class START,END terminal
    class prepare,process,format worker
    class summarize aggregator
    class agent subgraphStyle
    class outer subgraphStyle
```

**Shared state:** поля `topic` и `formatted` присутствуют и в `OuterState`, и в `InnerState` — LangGraph автоматически передаёт их между графами.

In [1]:
import sys
sys.path.insert(0, "..")

from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langchain_core.messages import HumanMessage, SystemMessage
from llm_config import get_llm, check_api_key

In [2]:
if not check_api_key():
    raise ValueError("API key is not set")
else:
    print("API key is set")

API key is set


## Состояние внутреннего графа

Внутренний граф — это самостоятельный агент с собственным состоянием `InnerState`. Он содержит три поля: `topic` (входная тема), `processed` (результат анализа) и `formatted` (отформатированный вывод). Поля `topic` и `formatted` совпадают по имени с полями внешнего графа — именно через них LangGraph передаёт данные между уровнями.

In [3]:
llm = get_llm()

class InnerState(TypedDict):
    topic: str
    processed: str
    formatted: str

## Узлы внутреннего графа

Внутренний агент состоит из двух шагов. `process_node` получает тему и просит LLM выделить три ключевых аспекта. `format_node` берёт результат анализа и оформляет его в виде нумерованного списка. Каждый узел обновляет только своё поле в состоянии.

In [4]:
def process_node(state: InnerState) -> dict:
    """Внутренний шаг 1: анализ темы."""
    response = llm.invoke([
        SystemMessage(content="Ты — аналитик. Выдели 3 ключевых аспекта темы. Кратко."),
        HumanMessage(content=state["topic"])
    ])
    print(f"  [Внутренний: process] Анализ: {response.content[:60]}...")
    return {"processed": response.content}


def format_node(state: InnerState) -> dict:
    """Внутренний шаг 2: форматирование результата."""
    response = llm.invoke([
        SystemMessage(content="Ты — форматировщик. Оформи текст как нумерованный список."),
        HumanMessage(content=state["processed"])
    ])
    print(f"  [Внутренний: format] Форматирование: {response.content[:60]}...")
    return {"formatted": response.content}

## Сборка и компиляция внутреннего графа

Внутренний граф собирается как обычный `StateGraph` и компилируется в самостоятельный объект. После компиляции он становится «чёрным ящиком», который можно использовать как узел в любом другом графе.

In [5]:
inner_graph = StateGraph(InnerState)
inner_graph.add_node("process", process_node)
inner_graph.add_node("format", format_node)
inner_graph.add_edge(START, "process")
inner_graph.add_edge("process", "format")
inner_graph.add_edge("format", END)

# Компилируем — получаем самостоятельный граф
inner_compiled = inner_graph.compile()

## Состояние внешнего графа

`OuterState` содержит три поля. Два из них — `topic` и `formatted` — совпадают по имени с полями `InnerState`. Это позволяет LangGraph автоматически передавать данные между графами: при входе в подграф `topic` попадает внутрь, при выходе — `formatted` возвращается обратно. Поле `summary` принадлежит только внешнему графу и заполняется финальным узлом.

In [6]:
class OuterState(TypedDict):
    topic: str        # Совпадает с InnerState.topic
    formatted: str    # Совпадает с InnerState.formatted
    summary: str      # Только внешний граф

## Узлы внешнего графа

Внешний граф содержит два собственных узла. `prepare_node` нормализует входную тему (trim + capitalize). `summarize_node` получает отформатированный результат от подграфа и просит LLM сжать его в краткое резюме из двух предложений.

In [7]:
def prepare_node(state: OuterState) -> dict:
    """Внешний шаг: подготовка (например, нормализация темы)."""
    topic = state["topic"].strip().capitalize()
    print(f"  [Внешний: prepare] Тема подготовлена: {topic}")
    return {"topic": topic}


def summarize_node(state: OuterState) -> dict:
    """Внешний шаг: финальное резюме результатов подграфа."""
    response = llm.invoke([
        SystemMessage(content="Ты — редактор. Напиши краткое резюме (2 предложения) на основе текста."),
        HumanMessage(content=state["formatted"])
    ])
    print(f"  [Внешний: summarize] Резюме: {response.content[:60]}...")
    return {"summary": response.content}

## Сборка внешнего графа с подграфом

Ключевой момент — строка `outer_graph.add_node("agent", inner_compiled)`. Скомпилированный внутренний граф передаётся как обычный узел. LangGraph автоматически сопоставляет поля состояний: если `InnerState` и `OuterState` имеют одноимённые поля, данные передаются между ними. Это позволяет подграфу работать с подмножеством полей родительского состояния, не зная о его полной структуре.

In [8]:
outer_graph = StateGraph(OuterState)

outer_graph.add_node("prepare", prepare_node)
outer_graph.add_node("agent", inner_compiled)      # Подграф как узел!
outer_graph.add_node("summarize", summarize_node)

outer_graph.add_edge(START, "prepare")
outer_graph.add_edge("prepare", "agent")            # Данные передаются автоматически
outer_graph.add_edge("agent", "summarize")
outer_graph.add_edge("summarize", END)

app = outer_graph.compile()

## Запуск

При вызове `app.invoke()` данные проходят через четыре шага: `prepare` (внешний) -> `process` (внутренний) -> `format` (внутренний) -> `summarize` (внешний). Обратите внимание на порядок логов — сначала внешний prepare, затем оба внутренних шага, и наконец внешний summarize.

In [9]:
result = app.invoke({
    "topic": "мультиагентные системы в production",
    "formatted": "",
    "summary": "",
})

print(f"\n  Результат подграфа:\n  {result['formatted'][:200]}...")
print(f"\n  Финальное резюме:\n  {result['summary'][:200]}")

  [Внешний: prepare] Тема подготовлена: Мультиагентные системы в production


  [Внутренний: process] Анализ: 1. **Оркестрация и роли агентов**  
   Чёткое разделение фун...


  [Внутренний: format] Форматирование: 1. **Оркестрация и роли агентов**  
   Чёткое разделение фун...


  [Внешний: summarize] Резюме: Текст описывает три ключевые задачи при построении мультиаге...

  Результат подграфа:
  1. **Оркестрация и роли агентов**  
   Чёткое разделение функций между агентами, управление их взаимодействием и предотвращение конфликтов.

2. **Надёжность и контроль качества**  
   Нужны валидация ...

  Финальное резюме:
  Текст описывает три ключевые задачи при построении мультиагентных систем: чёткую оркестрацию ролей и взаимодействий, обеспечение надёжности через валидацию и fallback-механизмы, а также готовность к p


## Итоги

- **Инкапсуляция**: подграф имеет собственное состояние и логику; родительский граф не знает о его внутренней структуре
- **Shared state**: одноимённые поля (`topic`, `formatted`) автоматически сопоставляются между `OuterState` и `InnerState` без ручного маппинга
- **Компонуемость**: скомпилированный граф — это обычный объект, который можно передать как узел в `add_node()`, переиспользовать в разных родительских графах или заменить другой реализацией
- **Масштабирование**: подграфы позволяют строить иерархические мультиагентные системы — каждый агент может быть сложным графом с собственной логикой